# Notebook 36 - Hardware cost proxy: single-thread CPU latency, memory and size

No edge board was available, so cost is measured on one CPU thread of the runtime and reported as a proxy, beside the paper's realised-FLOP figures. It is still a measurement of the actual pruned graphs rather than a count: the induced downstream reductions and the normalisation overheads that FLOP counting can miss show up here.

**Models.** Both corpora, both architectures: the dense teacher and the five frozen 40% students, rebuilt from the frozen structures with random weights, since latency, memory and size depend on the graph and not on the weight values.

**Measurements.** Median and 99th-percentile per-batch wall time over 200 timed passes after 20 warm-up passes, at batch sizes 1 and 256; parameter and activation footprint at batch 256; serialised size; realised FLOPs from the surgery profiler. Ratios are taken against each corpus-architecture's dense model. Descriptive, no pass or fail claim.

**Runtime.** CPU runtime is sufficient; a GPU is not used. A few minutes.

In [ ]:
# Stage 1 - bootstrap (CPU runtime is sufficient; a GPU is not used)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, time, gc, platform, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError as e:                     # already started; intra-op single-threading is what matters here
    print("interop threads left at default:", e)
R = REPO / "results/saber"; OUT = R / "36_hardware_proxy"; OUT.mkdir(parents=True, exist_ok=True)
CPU = platform.processor() or platform.machine()
try:
    CPU = [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
except Exception:
    pass
print("threads:", torch.get_num_threads(), "| cpu:", CPU, "| torch:", torch.__version__)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "H_hardware_proxy",
    "device": "single CPU thread of the Colab runtime (x86), reported as a PROXY for constrained hardware; no edge board was available",
    "models": ("both corpora, both architectures: the dense teacher and the five frozen 40% students (CICIoT2023: 17b/20b registries; "
               "IoMT: NB33 registry), rebuilt from the frozen structures with random weights, since latency and memory depend on the graph, not the values"),
    "measurements": {
        "latency": "median and 99th percentile of per-batch wall time over 200 timed forward passes after 20 warm-up passes, for batch sizes 1 and 256, eval mode, no grad",
        "memory": "parameter and buffer bytes, plus the summed bytes of every module output in one batch-256 forward pass captured by forward hooks (an upper bound on live activations)",
        "size": "serialised state_dict size in KB",
        "flops": "realised multiply-accumulates per flow from the surgery profiler, the paper's existing cost axis"},
    "reporting": "descriptive; latency reduction ratios placed beside realised-FLOP reduction ratios for the same models; no pass/fail claim",
    "no_test_access": True,
}
(OUT / "H_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2)); print(json.dumps(PREREG, indent=2))
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]; MIN_W = 8
WARMUP, TIMED = 20, 200


In [ ]:
# Stage 3 - model factories and frozen structures for both corpora
def make_models(n_classes):
    class CNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64), nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    class DeepCNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            def blk(i, o):
                return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
            self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    return {"shallow": CNN1D, "deep": DeepCNN1D}


CORPORA = {
    "ciciot2023": {"n_classes": 34, "n_features": 41,
                   "structure": lambda arch, m: (R / f"17b_calibrated_checkpoint_freeze/{m}_r40cal_removed_groups.csv") if arch == "shallow"
                                                 else (R / f"20b_depth_checkpoint_freeze/{m}_minimal_r40_removed_groups.csv")},
    "iomt2024":   {"n_classes": 19, "n_features": 45,
                   "structure": lambda arch, m: R / f"33_iomt_scores_structures/{arch}_{m}_r40_removed_groups.csv"},
}
for c, spec in CORPORA.items():
    for arch in ("shallow", "deep"):
        for m in METHODS:
            assert spec["structure"](arch, m).exists(), f"missing structure {c}/{arch}/{m}"
print("all 20 frozen structures present")


def build(corpus, arch, method=None):
    spec = CORPORA[corpus]; torch.manual_seed(0)
    dense = make_models(spec["n_classes"])[arch]().eval()
    example = torch.randn(8, spec["n_features"])
    if method is None:
        return dense, example
    rm = pd.read_csv(spec["structure"](arch, method))
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(dense, pm, example, minimum_remaining_per_layer=MIN_W)
    return st.eval(), example


In [ ]:
# Stage 4 - measurement
def footprint_mb(model, n_features, bs=256):
    """Parameter+buffer bytes plus the summed bytes of every module output in one forward pass (an upper bound on live activations)."""
    acts = []
    def hook(_m, _i, o):
        if torch.is_tensor(o):
            acts.append(o.numel() * o.element_size())
    hs = [m.register_forward_hook(hook) for m in model.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        model(torch.randn(bs, n_features))
    for h in hs:
        h.remove()
    params = sum(p.numel() * p.element_size() for p in model.parameters()) + sum(b.numel() * b.element_size() for b in model.buffers())
    return params / 1e6, sum(acts) / 1e6


def measure(model, n_features):
    out = {}
    out["params_mb"], out["activations_mb_b256"] = footprint_mb(model, n_features)
    with torch.no_grad():
        for bs in (1, 256):
            x = torch.randn(bs, n_features)
            for _ in range(WARMUP):
                model(x)
            t = []
            for _ in range(TIMED):
                t0 = time.perf_counter(); model(x); t.append(time.perf_counter() - t0)
            t = np.array(t) * 1e3
            out[f"latency_ms_b{bs}_median"] = float(np.median(t)); out[f"latency_ms_b{bs}_p99"] = float(np.percentile(t, 99))
    sd = model.state_dict(); tmp = OUT / "_tmp.pt"; torch.save(sd, tmp); out["size_kb"] = tmp.stat().st_size / 1024.0; tmp.unlink()
    return out


rows = []
for corpus, spec in CORPORA.items():
    for arch in ("shallow", "deep"):
        for method in [None] + METHODS:
            model, example = build(corpus, arch, method)
            flops = profile_forward_flops(model, example)["flops_per_item"]
            m = measure(model, spec["n_features"])
            rows.append({"corpus": corpus, "architecture": arch, "model": "dense" if method is None else method,
                         "parameters": int(count_parameters(model)), "flops_per_flow": float(flops), **m})
            print(f"{corpus} {arch} {rows[-1]['model']:9s}: params {rows[-1]['parameters']:7d} | flops {flops:9.0f} | "
                  f"b1 {m['latency_ms_b1_median']:.3f} ms (p99 {m['latency_ms_b1_p99']:.3f}) | b256 {m['latency_ms_b256_median']:.2f} ms | size {m['size_kb']:.0f} KB")
            del model; gc.collect()
res = pd.DataFrame(rows)
# ratios relative to each corpus/architecture's dense model
for key in ["flops_per_flow", "latency_ms_b1_median", "latency_ms_b256_median", "size_kb", "parameters", "activations_mb_b256"]:
    dense = res[res.model == "dense"].set_index(["corpus", "architecture"])[key]
    res[f"{key}_ratio"] = res.apply(lambda r: r[key] / dense.loc[(r.corpus, r.architecture)], axis=1)
res.to_csv(OUT / "hardware_proxy.csv", index=False)
print("\nreduction ratios (student / dense), realised FLOPs vs latency:")
print(res[res.model != "dense"][["corpus", "architecture", "model", "flops_per_flow_ratio", "latency_ms_b1_median_ratio", "latency_ms_b256_median_ratio", "size_kb_ratio", "activations_mb_b256_ratio"]].round(3).to_string(index=False))


In [ ]:
# Stage 5 - record and figure
res = pd.read_csv(OUT / "hardware_proxy.csv")
students = res[res.model != "dense"]
summary = {"arm": "H_hardware_proxy", "cpu": CPU, "torch": torch.__version__, "threads": int(torch.get_num_threads()),
           "flops_ratio_range": [float(students.flops_per_flow_ratio.min()), float(students.flops_per_flow_ratio.max())],
           "latency_b1_ratio_range": [float(students.latency_ms_b1_median_ratio.min()), float(students.latency_ms_b1_median_ratio.max())],
           "latency_b256_ratio_range": [float(students.latency_ms_b256_median_ratio.min()), float(students.latency_ms_b256_median_ratio.max())],
           "size_ratio_range": [float(students.size_kb_ratio.min()), float(students.size_kb_ratio.max())],
           "activation_ratio_range_b256": [float(students.activations_mb_b256_ratio.min()), float(students.activations_mb_b256_ratio.max())],
           "dense_latency_ms_b1": {f"{r.corpus}/{r.architecture}": round(float(r.latency_ms_b1_median), 3) for r in res[res.model == "dense"].itertuples()},
           "dense_size_kb": {f"{r.corpus}/{r.architecture}": round(float(r.size_kb), 1) for r in res[res.model == "dense"].itertuples()},
           "note": "single-thread x86 proxy; per-flow latency at batch 1 is dominated by framework overhead at this model size, which is why the b256 ratio tracks FLOPs more closely",
           "prereg": json.load(open(OUT / "H_PREREGISTRATION.json"))}
(OUT / "H_summary.json").write_text(json.dumps(summary, indent=2)); print(json.dumps({k: v for k, v in summary.items() if k != "prereg"}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.6))
for ax, key, lab in [(axes[0], "latency_ms_b1_median_ratio", "latency ratio, batch 1"), (axes[1], "latency_ms_b256_median_ratio", "latency ratio, batch 256")]:
    for (c, a), g in students.groupby(["corpus", "architecture"]):
        ax.scatter(g.flops_per_flow_ratio, g[key], label=f"{c} {a}", s=28)
    lim = [0.4, 1.05]; ax.plot(lim, lim, color="0.6", lw=0.8, ls=":"); ax.set_xlim(lim); ax.set_xlabel("realised FLOP ratio (student / dense)"); ax.set_ylabel(lab)
    ax.set_title("Single-thread CPU: " + lab)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(OUT / "H_flops_vs_latency.png", dpi=200); plt.show()
print("written ->", OUT)
